# Odysseus GPU LLM + Cloudflare Tunnel

Downloads the 4-bit Qwen3.6-35B-A3B GGUF and serves it through an authenticated OpenAI-compatible API on Kaggle's two T4 GPUs. A temporary Cloudflare Quick Tunnel lets Odysseus reach the server. Keep this notebook private and running while you use the model. Kaggle restarts create a new tunnel URL and API key.

Before running: choose **GPU T4 x2** under **Settings → Accelerator** and make sure **Internet** is on.

In [ ]:
import os, re, secrets, subprocess, sys, time
from pathlib import Path
import requests, torch

if torch.cuda.device_count() < 2:
    raise RuntimeError(f'Expected Kaggle GPU T4 x2; found {torch.cuda.device_count()} GPU(s). Set Settings → Accelerator to GPU T4 x2 and restart.')
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

WORK = Path('/kaggle/working')
KEY_FILE = Path('/tmp/odysseus-api-key')
API_KEY = secrets.token_urlsafe(36)
KEY_FILE.write_text(API_KEY)
KEY_FILE.chmod(0o600)
os.environ['ODYSSEUS_API_KEY_FILE'] = str(KEY_FILE)
os.environ['HF_HOME'] = str(WORK / '.cache' / 'huggingface')

script_url = 'https://raw.githubusercontent.com/Mhdkenzz/odysseus/dev/scripts/kaggle_llm_server.py'
script_path = WORK / 'kaggle_llm_server.py'
response = requests.get(script_url, timeout=30)
response.raise_for_status()
script_path.write_bytes(response.content)

server_log_path = WORK / 'odysseus-llm.log'
server_log = open(server_log_path, 'w')
server = subprocess.Popen([sys.executable, str(script_path)], stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
print('Downloading the official CUDA server binaries and about 16.6 GB of Qwen weights; then loading the model. This can take a while.', flush=True)

deadline = time.time() + 7200
log_position = 0
while time.time() < deadline:
    if server.poll() is not None:
        print(server_log_path.read_text(errors='ignore')[-10000:])
        raise RuntimeError('Qwen server exited; see the log above.')
    try:
        health = requests.get('http://127.0.0.1:8000/health', timeout=4)
        if health.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(15)
    content = server_log_path.read_text(errors='ignore')
    new_text = content[log_position:]
    if new_text.strip():
        print(new_text[-2500:], flush=True)
    log_position = len(content)
else:
    print(server_log_path.read_text(errors='ignore')[-10000:])
    raise TimeoutError('Model did not become ready within 2 hours.')

print('Model is ready; starting the authenticated Cloudflare Tunnel.', flush=True)
cloudflared = WORK / 'cloudflared'
subprocess.run(['wget', '-q', '-O', str(cloudflared), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
cloudflared.chmod(0o755)
tunnel_log = open(WORK / 'cloudflared.log', 'w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)

tunnel_deadline = time.time() + 180
base_url = None
while time.time() < tunnel_deadline:
    log_text = Path(WORK / 'cloudflared.log').read_text(errors='ignore')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        base_url = match.group(0) + '/v1'
        break
    if tunnel.poll() is not None:
        raise RuntimeError('Cloudflare Tunnel exited; inspect /kaggle/working/cloudflared.log')
    time.sleep(2)
if not base_url:
    raise TimeoutError('Tunnel URL did not appear; inspect /kaggle/working/cloudflared.log')

models = None
last_error = None
models_deadline = time.time() + 180
while time.time() < models_deadline:
    try:
        models = requests.get(base_url + '/models', headers={'Authorization': 'Bearer ' + API_KEY}, timeout=15)
        models.raise_for_status()
        break
    except requests.RequestException as error:
        last_error = error
        print('Waiting for the new Cloudflare address to become reachable...', flush=True)
        time.sleep(5)
if models is None:
    raise RuntimeError('The Cloudflare URL did not become reachable; rerun this cell to create a fresh tunnel.') from last_error
print('READY — add the endpoint in Odysseus Settings → Models.')
print('Base URL:', base_url)
print('Model: Qwen/Qwen3.6-35B-A3B')
print('API key:', API_KEY)
